# 03 株式・暗号資産データ収集

| 項目 | 内容 |
|------|------|
| 目的 | yfinance を使って JP 株式・暗号資産・マクロ指標の OHLCV を収集 |
| 出力 | `data/equity_jp_ohlcv.parquet`, `data/crypto_ohlcv.parquet`, `data/macro.parquet` |
| 注意 | raw data は読み取り専用。前処理済みデータは processed_data/ に保存 |

---
## 0. 環境セットアップ

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import yaml

try:
    import japanize_matplotlib
except ImportError:
    pass

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

print('Setup OK')

Setup OK


---
## 1. 設定読み込み

In [2]:
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR      = Path(cfg['paths']['data'])
PROCESSED_DIR = Path(cfg['paths']['processed_data'])
FIGURES_DIR   = Path(cfg['paths']['figures'])
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_START     = cfg['equity']['data_start']
EQUITY_SYMBOLS = cfg['equity']['symbols']
CRYPTO_SYMBOLS = cfg['crypto']['symbols']
RANDOM_SEED    = cfg.get('random_seed', 42)

print(f'Data start : {DATA_START}')
print(f'Equity     : {len(EQUITY_SYMBOLS)} 銘柄')
print(f'Crypto     : {len(CRYPTO_SYMBOLS)} シンボル')
print(f'Data dir   : {DATA_DIR.resolve()}')

Data start : 2020-01-01
Equity     : 25 銘柄
Crypto     : 5 シンボル
Data dir   : C:\Users\hngka\Project\DataScience\StockPrediction\Analytics\notebooks\data


---
## 2. 日本株 OHLCV 収集

In [3]:
from datetime import datetime

def download_ohlcv(symbols, start, end=None, asset_class='equity_jp'):
    if end is None:
        end = datetime.today().strftime('%Y-%m-%d')
    print(f'ダウンロード: {len(symbols)} 銘柄  {start} ~ {end}')
    raw = yf.download(symbols, start=start, end=end, auto_adjust=True, progress=True)
    records = []
    for sym in symbols:
        try:
            df_s = raw.xs(sym, level=1, axis=1).copy() if len(symbols) > 1 else raw.copy()
            df_s = df_s.dropna(subset=['Close'])
            if df_s.empty:
                continue
            df_s['symbol'] = sym
            df_s.index.name = 'date'
            df_s = df_s.reset_index()
            df_s.columns = [c.lower() if c != 'date' else c for c in df_s.columns]
            records.append(df_s)
        except Exception as e:
            print(f'  WARN [{sym}]: {e}')
    if not records:
        return pd.DataFrame()
    df = pd.concat(records, ignore_index=True)
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
    df['asset_class'] = asset_class
    cols = ['symbol', 'asset_class', 'date', 'open', 'high', 'low', 'close', 'volume']
    return df[cols].sort_values(['symbol', 'date']).reset_index(drop=True)


df_equity = download_ohlcv(EQUITY_SYMBOLS, start=DATA_START)
print(f'\n行数   : {len(df_equity):,}')
print(f'銘柄数 : {df_equity["symbol"].nunique()}')
print(f'期間   : {df_equity["date"].min().date()} ~ {df_equity["date"].max().date()}')
df_equity.head()

[                       0%%                      ]

ダウンロード: 25 銘柄  2020-01-01 ~ 2026-07-09


Failed to get ticker '4502.T' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker '9984.T' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker '8306.T' reason: Expecting value: line 1 column 1 (char 0)
[******                12%%                      ]  3 of 25 completedFailed to get ticker '6758.T' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker '4568.T' reason: Expecting value: line 1 column 1 (char 0)
[********              16%%                      ]  4 of 25 completedFailed to get ticker '9433.T' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker '6702.T' reason: Expecting value: line 1 column 1 (char 0)
[************          24%%                      ]  6 of 25 completedFailed to get ticker '6501.T' reason: Expecting value: line 1 column 1 (char 0)
[*****************     36%%                      ]  9 of 25 completedFailed to get ticker '6503.T' reason: Expecting value: line 1 column 1 (char 0)
[***


行数   : 0


KeyError: 'symbol'

---
## 3. データ品質チェック

In [ ]:
print('=== 基本情報 ===')
print(df_equity.dtypes)

print('\n=== 欠損値 ===')
missing = df_equity[['open','high','low','close','volume']].isnull().sum()
print(missing[missing > 0] if (missing > 0).any() else '  欠損なし')

print('\n=== 銘柄別日数 (下位 5 件) ===')
by_sym = df_equity.groupby('symbol').size().rename('days').reset_index()
print(by_sym.sort_values('days').head())

print('\n=== 統計量 (close) ===')
print(df_equity.groupby('symbol')['close'].describe().T)

In [ ]:
# 代表 6 銘柄のチャート
sample = EQUITY_SYMBOLS[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, sym in zip(axes.flatten(), sample):
    df_s = df_equity[df_equity['symbol'] == sym]
    ax.plot(df_s['date'], df_s['close'], linewidth=0.8, color='steelblue')
    ax.set_title(sym, fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(alpha=0.3)
plt.suptitle('JP 株式 終値チャート', fontsize=14)
plt.tight_layout()
path = FIGURES_DIR / '03_equity_price_chart.png'
plt.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'保存: {path}')

---
## 4. 暗号資産 OHLCV 収集

In [ ]:
df_crypto = download_ohlcv(CRYPTO_SYMBOLS, start=DATA_START, asset_class='crypto')
print(f'行数   : {len(df_crypto):,}')
print(f'期間   : {df_crypto["date"].min().date()} ~ {df_crypto["date"].max().date()}')
df_crypto.head()

---
## 5. マクロ指標収集

In [ ]:
MACRO_TICKERS = cfg.get('macro_tickers', {
    '^N225':    'nikkei225',
    'DX-Y.NYB': 'usd_index',
    'USDJPY=X': 'usdjpy',
    '^TNX':     'us10y_yield',
    'CL=F':     'crude_oil',
    'GC=F':     'gold',
})

series_list = []
for ticker, name in MACRO_TICKERS.items():
    try:
        s = yf.download(ticker, start=DATA_START, auto_adjust=True, progress=False)['Close']
        s.name = name
        series_list.append(s)
        print(f'  OK: {ticker:15s} -> {name}')
    except Exception as e:
        print(f'  WARN: {ticker}: {e}')

df_macro = pd.concat(series_list, axis=1).reset_index()
df_macro.columns = ['date'] + [c for c in df_macro.columns if c != 'date']
df_macro['date'] = pd.to_datetime(df_macro['date']).dt.tz_localize(None)
df_macro = df_macro.sort_values('date').set_index('date').ffill().reset_index()
print(f'\nマクロ: {len(df_macro):,} 行 x {df_macro.shape[1]-1} 指標')
df_macro.tail()

In [ ]:
# マクロ指標の正規化プロット
macro_cols = [c for c in df_macro.columns if c != 'date']
fig, ax = plt.subplots(figsize=(14, 5))
for col in macro_cols:
    s = df_macro[col].dropna()
    s_norm = (s - s.mean()) / s.std()
    ax.plot(df_macro.loc[s.index, 'date'], s_norm, label=col, linewidth=0.8)
ax.set_title('マクロ指標 (Z スコア正規化)', fontsize=13)
ax.legend(ncol=3, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. データ保存

In [ ]:
df_equity.to_parquet(DATA_DIR / 'equity_jp_ohlcv.parquet', index=False)
df_crypto.to_parquet(DATA_DIR / 'crypto_ohlcv.parquet', index=False)
df_macro.to_parquet(DATA_DIR / 'macro.parquet', index=False)

print('保存完了:')
print(f'  equity_jp_ohlcv.parquet  {len(df_equity):>8,} 行')
print(f'  crypto_ohlcv.parquet     {len(df_crypto):>8,} 行')
print(f'  macro.parquet            {len(df_macro):>8,} 行')
print()
print('次のステップ: 42_equity_lgbm_walkforward.ipynb でモデル構築')